In [2]:
import pandas as pd
data = pd.read_csv("sauto_dataset.csv", sep=',')


data.loc[data["pohon"].isna(), "pohon"] = "nezadano"
# maska_ke_smazani = (data['palivo'] == 'elektro') & ((data['kapacita baterie'] == 0) | (data['kapacita baterie'].isna()))
# data = data[~maska_ke_smazani]

indexy_elektro = data[(data['palivo'] == 'Elektro') & (data['kapacita_baterie'] == 0)].index
indexy_none = data[(data['znacka'].isna()) | (data['model'].isna()) | (data['cena'].isna()) | (data['rok'].isna()) | (data['stav'].isna()) | (data['najeto'].isna()) | (data['vykon'].isna()) | (data['palivo'].isna()) | (data['prevodovka'].isna())].index

vsechny_ke_smazani = indexy_elektro.union(indexy_none)
# vsechny_ke_smazani = indexy_elektro
data = data.drop(vsechny_ke_smazani)



string_features = ['znacka', 'model', 'stav', 'palivo', 'prevodovka', 'pohon']
mapping = {}
for feature in string_features:
    data[feature] = data[feature].astype('category')
    mapping[feature] = list(data[feature].cat.categories)
    data[feature] = data[feature].cat.codes

# Jak zjistím, co je co?
# print(mapping['znacka'])
print(mapping['stav'])
print(mapping['palivo'])
print(mapping['prevodovka'])
print(mapping['pohon'])

kod = mapping["znacka"].index('Tesla')
print(f"znacka: ",kod)
kod = mapping["model"].index('Model 3')
print(f"model: ",kod)




# maska = (data["palivo"] == "Elektro") & (data["kapacita_baterie"] == 0)
# print(data[maska])# data

# idk = (data["palivo"] == "Elektro")
# print(data[idk])
# print(data["znacka"].unique())

data
import json



['Nové', 'Ojeté', 'Předváděcí']
['Benzín', 'CNG + benzín', 'Elektro', 'Hybridní', 'LPG + benzín', 'Nafta']
['Automatická', 'Manuální', 'Poloautomatická']
['4x4', 'Pohon předních kol', 'Pohon zadních kol', 'nezadano']
znacka:  74
model:  496


In [ ]:
# Uložení slovníku s kódováním pro konzolovou aplikaci
with open("mapping.json", "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=4)
print("Mapping byl úspěšně uložen do mapping.json")

In [3]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.tree import DecisionTreeRegressor




input_features = ["znacka", "model", "rok", "stav", "najeto", "vykon", "palivo", "kapacita_baterie", "prevodovka", "pohon"]
target_feature = 'cena'

X_train, X_test, y_train, y_test = train_test_split(data[input_features], data[target_feature], test_size=0.1, random_state=42)
# X_train vstupni trenovaci
# X_test vystupni trenovaci
# y_test vstupni testovaci
# y_train vustupni testovaci


from sklearn.ensemble import GradientBoostingRegressor

# Definice modelu
model = GradientBoostingRegressor(
    n_estimators=500,     # Počet stromů (u boosting se jich dává víc než u RF)
    learning_rate=0.1,    # Rychlost učení (klíčový parametr pro boosting)
    max_depth=5,          # Stromy u boostingu jsou obvykle mělčí (např. 3 až 8)
    min_samples_split=5,
    random_state=42
)

# Trénování i predikce zůstávají stejné
model.fit(X_train, y_train)
y_pred = model.predict(X_test)



y_pred = model.predict(X_test)


In [4]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(mae)
print(mse)

83947.24142899545
159574442428.0118


In [2]:
from sklearn.tree import export_text
tree_rules = export_text(model, feature_names=input_features)
print(tree_rules)


NameError: name 'model' is not defined

In [1]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
plt.figure(figsize=(18, 10))
plot_tree(
    model,
    feature_names=input_features,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Regressor")
plt.show()

NameError: name 'model' is not defined

<Figure size 1800x1000 with 0 Axes>

In [21]:
print(model.tree_.threshold)

[ 4.91499996e+01  2.01650000e+03  2.01250000e+03  2.00950000e+03
  3.77999992e+01  2.00750000e+03 -2.00000000e+00 -2.00000000e+00
  2.00650000e+03 -2.00000000e+00 -2.00000000e+00  4.75000000e+01
  2.01150000e+03 -2.00000000e+00 -2.00000000e+00 -2.00000000e+00
  2.01550000e+03  3.18510000e+04  3.08870000e+04 -2.00000000e+00
 -2.00000000e+00  1.05000000e+01 -2.00000000e+00 -2.00000000e+00
  4.67500000e+01  4.19499989e+01 -2.00000000e+00 -2.00000000e+00
  8.00000000e+00 -2.00000000e+00 -2.00000000e+00  1.80000001e+00
  4.51999989e+01  5.00000000e-01  4.12000008e+01 -2.00000000e+00
 -2.00000000e+00  4.01000004e+01 -2.00000000e+00 -2.00000000e+00
  1.55000001e+00  1.50000000e+00 -2.00000000e+00 -2.00000000e+00
  1.36050000e+03 -2.00000000e+00 -2.00000000e+00  2.01850000e+03
  4.25000000e+01  3.97820000e+04 -2.00000000e+00 -2.00000000e+00
  2.41120000e+04 -2.00000000e+00 -2.00000000e+00  4.50000000e+00
  4.18500004e+01 -2.00000000e+00 -2.00000000e+00  7.00000000e+00
 -2.00000000e+00 -2.00000

In [5]:
import pickle

with open("model_sautoV4.dat", "wb") as soubor:
     pickle.dump(model, soubor)